In [1]:
# ════════════════════════════════════════════════════════
# CELL 1 — Kaggle paths + create folders
# ════════════════════════════════════════════════════════
import os

# Your dataset will be mounted here automatically on Kaggle
# when you add it as a dataset input in the notebook settings

DATASET_ROOT = '/kaggle/input/datasets/abhinavkishan123/deepfashion-inshop-dataset/Dataset'
# ↑ Replace 'your-dataset-name' with whatever slug Kaggle gives
# when you upload — check the URL, e.g. /kaggle/input/deepfashion-inshop

# All outputs go here — persists within the session
PROJ = '/kaggle/working'

DIRS = [
    f'{PROJ}/crops/gt',
    f'{PROJ}/crops/yolo',
    f'{PROJ}/captions',
    f'{PROJ}/checkpoints',
    f'{PROJ}/index',
    f'{PROJ}/results',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Working dir :", PROJ)
print("Folders created:")
for d in DIRS: print(f"  {d}")

Dataset root: /kaggle/input/datasets/abhinavkishan123/deepfashion-inshop-dataset/Dataset
Working dir : /kaggle/working
Folders created:
  /kaggle/working/crops/gt
  /kaggle/working/crops/yolo
  /kaggle/working/captions
  /kaggle/working/checkpoints
  /kaggle/working/index
  /kaggle/working/results


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — Install all dependencies (run once)
# ════════════════════════════════════════════════════════
!pip install ultralytics open-clip-torch hnswlib \
             transformers accelerate -q

print("All packages installed.")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — Parse your exact files
# (matching what's visible in your screenshot)
# ════════════════════════════════════════════════════════
import pandas as pd

# ── Your exact file paths ────────────────────────────────
EVAL_PARTITION = f'{DATASET_ROOT}/list_eval_partition.txt'
BBOX_FILE      = f'{DATASET_ROOT}/list_bbox_inshop.txt'
# NOTE: list_description_inshop.json is IGNORED
# TA confirmed: use BLIP-2 to generate descriptions, not this file

IMG_DIR = f'{DATASET_ROOT}/img/img'   # the img/ folder in your screenshot

# Verify files exist
for path in [EVAL_PARTITION, BBOX_FILE, IMG_DIR]:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path}")

# ── Parse eval partition ─────────────────────────────────
def parse_eval_partition(path):
    with open(path) as f:
        lines = f.read().strip().split('\n')
    n = int(lines[0])
    rows = []
    for line in lines[2: 2 + n]:
        parts = line.split()
        rows.append({'image_path': parts[0], 'split': parts[1]})
    return pd.DataFrame(rows)

# ── Parse bbox file ──────────────────────────────────────
def parse_bbox_list(path):
    with open(path) as f:
        lines = f.read().strip().split('\n')
    n = int(lines[0])
    rows = []
    for line in lines[2: 2 + n]:
        parts = line.split()
        rows.append({
            'image_path': parts[0],
            'x1': int(parts[3]),
            'y1': int(parts[4]),
            'x2': int(parts[5]),
            'y2': int(parts[6]),
        })
    return pd.DataFrame(rows)

# ── No list_item.txt in your folder ──────────────────────
# DeepFashion item_id is encoded IN the image path itself.
# Path format: img/category/id_XXXXXX/img_name.jpg
# We extract item_id from the path directly.
def extract_item_id(image_path):
    # e.g. "img/WOMEN/Blouses_Shirts/id_00000001/01_1_front.jpg"
    #                                 ^^^^^^^^^^^
    parts = image_path.replace('\\', '/').split('/')
    for part in parts:
        if part.startswith('id_'):
            return part
    return image_path   # fallback

eval_df = parse_eval_partition(EVAL_PARTITION)
bbox_df = parse_bbox_list(BBOX_FILE)

# Add item_id extracted from path
eval_df['item_id'] = eval_df['image_path'].apply(extract_item_id)

# Merge bbox
df = eval_df.merge(bbox_df, on='image_path', how='left')

print("\nSplit counts:")
print(df['split'].value_counts())
print(f"\nUnique items: {df['item_id'].nunique():,}")
print(f"Missing bbox: {df['x1'].isna().sum():,}")
print(df.head(3))

df.to_csv(f'{PROJ}/dataset_index.csv', index=False)
print(f"\nSaved → {PROJ}/dataset_index.csv")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — Parse annotation files (fixed)
# list_eval_partition.txt has 3 columns:
#   image_name | item_id | evaluation_status
# ════════════════════════════════════════════════════════
import pandas as pd

def parse_eval_partition(path):
    """
    Format: image_name   item_id   evaluation_status
    parts[0] = image_path
    parts[1] = item_id        ← was being read as split before (BUG)
    parts[2] = split label    ← train / query / gallery
    """
    with open(path) as f:
        lines = f.read().strip().split('\n')
    n = int(lines[0])
    rows = []
    for line in lines[2: 2 + n]:
        parts = line.split()
        rows.append({
            'image_path': parts[0],
            'item_id':    parts[1],   # directly from file, no extraction needed
            'split':      parts[2],   # train / query / gallery
        })
    return pd.DataFrame(rows)

def parse_bbox_list(path):
    with open(path) as f:
        lines = f.read().strip().split('\n')
    n = int(lines[0])
    rows = []
    for line in lines[2: 2 + n]:
        parts = line.split()
        rows.append({
            'image_path': parts[0],
            'x1': int(parts[3]),
            'y1': int(parts[4]),
            'x2': int(parts[5]),
            'y2': int(parts[6]),
        })
    return pd.DataFrame(rows)

# ── Load ─────────────────────────────────────────────────
eval_df = parse_eval_partition(f'{DATASET_ROOT}/list_eval_partition.txt')
bbox_df = parse_bbox_list(f'{DATASET_ROOT}/list_bbox_inshop.txt')

# ── Merge bbox ───────────────────────────────────────────
df = eval_df.merge(bbox_df, on='image_path', how='left')

# ── Verify ───────────────────────────────────────────────
print("Split counts:")
print(df['split'].value_counts())
print(f"\nTotal rows   : {len(df):,}")
print(f"Unique items : {df['item_id'].nunique():,}")
print(f"Missing bbox : {df['x1'].isna().sum():,}")
print("\nSample rows:")
print(df[['image_path', 'item_id', 'split', 'x1', 'y1', 'x2', 'y2']].head(5).to_string())

df.to_csv(f'{PROJ}/dataset_index.csv', index=False)
print(f"\nSaved → {PROJ}/dataset_index.csv")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — EDA: understand the dataset before touching models
# ════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

train_df   = df[df['split'] == 'train']
query_df   = df[df['split'] == 'query']
gallery_df = df[df['split'] == 'gallery']

print("="*45)
print(f"{'Split':<10} {'Images':>8} {'Unique items':>14}")
print("="*45)
for name, sub in [('train', train_df), ('query', query_df), ('gallery', gallery_df)]:
    print(f"{name:<10} {len(sub):>8,} {sub['item_id'].nunique():>14,}")
print("="*45)

# Images per item in train
counts = train_df.groupby('item_id').size()
print(f"\nTrain images/item — min:{counts.min()}  max:{counts.max()}  mean:{counts.mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(counts, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Images per item (train)')
axes[0].set_xlabel('# images'); axes[0].set_ylabel('# items')

# BBox coverage check
has_bbox = df['x1'].notna().sum()
axes[1].bar(['Has bbox', 'No bbox'], [has_bbox, len(df) - has_bbox], color=['#2ecc71','#e74c3c'])
axes[1].set_title('BBox annotation coverage')

plt.tight_layout()
plt.savefig(f'{PROJ}/eda.png', dpi=150)
plt.show()
print(f"BBox coverage: {has_bbox/len(df)*100:.1f}%")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — GT bbox cropping for TRAIN + GALLERY
#
# PDF says Step 1 = YOLO for localization.
# For offline indexing we use GT bboxes (cleaner, no detection error).
# YOLO is reserved for query images (realistic inference scenario).
# ════════════════════════════════════════════════════════
import json
from PIL import Image
from tqdm import tqdm

GT_CROP_DIR = f'{PROJ}/crops/gt'
crop_meta_gt = {}    # { relative_image_path : absolute_crop_path }

# Only train + gallery get GT crops
for_gt = df[df['split'].isin(['train', 'gallery'])].reset_index(drop=True)
print(f"Images to GT-crop: {len(for_gt):,}")

for _, row in tqdm(for_gt.iterrows(), total=len(for_gt), desc="GT cropping"):
    safe_name = row['image_path'].replace('/', '__')
    dst_path  = f"{GT_CROP_DIR}/{safe_name}"

    if os.path.exists(dst_path):
        crop_meta_gt[row['image_path']] = dst_path
        continue

    src_path = f"{DATASET_ROOT}/img/{row['image_path']}"
   # print(df['image_path'].iloc[0])
    try:
        img = Image.open(src_path).convert('RGB')
        W, H = img.size

        # Use GT bbox if available and valid
        if pd.notna(row['x1']):
            x1 = max(0, int(row['x1']))
            y1 = max(0, int(row['y1']))
            x2 = min(W, int(row['x2']))
            y2 = min(H, int(row['y2']))
            if x2 > x1 and y2 > y1:
                img = img.crop((x1, y1, x2, y2))
        # else: no bbox → keep full image (fallback)

        img.save(dst_path)
        crop_meta_gt[row['image_path']] = dst_path

    except Exception as e:
        # Save full image path as fallback so pipeline never breaks
        crop_meta_gt[row['image_path']] = src_path

with open(f'{PROJ}/crop_meta_gt.json', 'w') as f:
    json.dump(crop_meta_gt, f)

print(f"\nGT crops done: {len(crop_meta_gt):,}")
print(f"Saved → {PROJ}/crop_meta_gt.json")

In [ ]:
# Quick verify — run this to confirm crops are real
import os, json

crop_meta_gt = json.load(open(f'{PROJ}/crop_meta_gt.json'))
print(f"Total entries: {len(crop_meta_gt):,}")

# Check first 3 crops actually exist on disk
for path, crop_path in list(crop_meta_gt.items())[:3]:
    exists = os.path.exists(crop_path)
    size   = os.path.getsize(crop_path) if exists else 0
    print(f"{'✓' if exists else '✗'} {crop_path}  ({size} bytes)")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — YOLO cropping for QUERY images
#
# Query images simulate real user uploads at inference time.
# No GT bbox available at runtime → use YOLO, exactly like
# the online pipeline will do in the Streamlit demo.
# ════════════════════════════════════════════════════════
from ultralytics import YOLO

# Use yolov8n (lightest, fastest) — or swap in your mini-project weights
yolo = YOLO('yolov8n.pt')

YOLO_CROP_DIR = f'{PROJ}/crops/yolo'
crop_meta_yolo = {}

query_only = df[df['split'] == 'query'].reset_index(drop=True)
print(f"Query images to YOLO-crop: {len(query_only):,}")

for _, row in tqdm(query_only.iterrows(), total=len(query_only), desc="YOLO cropping queries"):
    safe_name = row['image_path'].replace('/', '__')
    dst_path  = f"{YOLO_CROP_DIR}/{safe_name}"

    if os.path.exists(dst_path):
        crop_meta_yolo[row['image_path']] = dst_path
        continue

    src_path = f"{DATASET_ROOT}/img/{row['image_path']}"
    try:
        img     = Image.open(src_path).convert('RGB')
        W, H    = img.size
        results = yolo(img, verbose=False)
        boxes   = results[0].boxes

        # Prefer class 0 (person) — clothing is worn by model in DeepFashion
        best_box, best_area = None, 0
        for i, cls in enumerate(boxes.cls.cpu().numpy()):
            if int(cls) == 0:
                x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
                area = (x2 - x1) * (y2 - y1)
                if area > best_area:
                    best_area = area
                    best_box  = (int(x1), int(y1), int(x2), int(y2))

        if best_box:
            img = img.crop(best_box)
        # else: no detection → keep full image

        img.save(dst_path)
        crop_meta_yolo[row['image_path']] = dst_path

    except Exception as e:
        crop_meta_yolo[row['image_path']] = src_path  # fallback

with open(f'{PROJ}/crop_meta_yolo.json', 'w') as f:
    json.dump(crop_meta_yolo, f)

print(f"\nYOLO query crops done: {len(crop_meta_yolo):,}")
print(f"Saved → {PROJ}/crop_meta_yolo.json")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — Visual sanity check: compare GT crop vs YOLO crop
# ════════════════════════════════════════════════════════
import random, matplotlib.pyplot as plt

# Pick a query image that also exists in gallery (same item_id)
sample_row = df[df['split'] == 'query'].sample(1).iloc[0]
src_path   = f"{DATASET_ROOT}/img/{sample_row['image_path']}"

# Full image
full_img = Image.open(src_path).convert('RGB')

# YOLO crop for this query image
yolo_path = crop_meta_yolo.get(sample_row['image_path'], src_path)
yolo_img  = Image.open(yolo_path).convert('RGB')

# A GT-cropped gallery image of the same item_id
same_item = df[(df['item_id'] == sample_row['item_id']) &
               (df['split'] == 'gallery')].iloc[0]
gt_path   = crop_meta_gt.get(same_item['image_path'], src_path)
gt_img    = Image.open(gt_path).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
axes[0].imshow(full_img);  axes[0].set_title('Full query image'); axes[0].axis('off')
axes[1].imshow(yolo_img);  axes[1].set_title('YOLO crop (query)'); axes[1].axis('off')
axes[2].imshow(gt_img);    axes[2].set_title(f'GT crop (gallery)\nitem_id={sample_row["item_id"]}'); axes[2].axis('off')
plt.tight_layout(); plt.show()
print("Sanity check passed ✓")

In [ ]:
import os
import shutil

# Path to working directory
source_dir = "/kaggle/working"
output_zip = "/kaggle/working/download_all"

# Create zip file
shutil.make_archive(output_zip, 'zip', source_dir)

print("Zipped successfully:", output_zip + ".zip")